In [47]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report,f1_score,roc_auc_score,roc_curve
from sklearn.preprocessing import StandardScaler,OrdinalEncoder,OneHotEncoder,LabelEncoder,PowerTransformer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier,Pool
import catboost
import optuna


In [48]:
df = pd.read_csv('final_data.csv')
df = df.drop(['education', 'housing', 'campaign_call', 'previous', 'top_jobs',
       'multiple_calls', 'previous_camp_call','contact', 'loan',
       'contacted_before', 'default', 'high_intensity_calls'],axis=1)
df.head()

,age,job,marital,balance,day,month,duration,campaign,pdays,poutcome,y,age_bin,balance_bins,loan_default_risk,duration_min,is_long_cal,pdays_cat,is_q2_calls,week,qtr
0,58,management,married,2143,5,may,261,1,-1,never contacted,no,56-65,high positive,1,less than 5 minutes,0,never contacted,1,week 1,Q2
1,44,technician,single,29,5,may,151,1,-1,never contacted,no,41-55,low positive,1,less than 3 minutes,0,never contacted,1,week 1,Q2
2,33,entrepreneur,married,2,5,may,76,1,-1,never contacted,no,31-40,low positive,1,less than 2 minutes,0,never contacted,1,week 1,Q2
3,47,blue-collar,married,1506,5,may,92,1,-1,never contacted,no,41-55,high positive,1,less than 2 minutes,0,never contacted,1,week 1,Q2
4,33,blue-collar,single,1,5,may,198,1,-1,never contacted,no,31-40,low positive,0,less than 4 minutes,0,never contacted,1,week 1,Q2


In [49]:
x = df.drop(['y'],axis=1)
y = df['y']

In [50]:
x[['job','marital','day','month','poutcome','loan_default_risk' ,'is_long_cal' ,'week' ,'qtr' ,'is_q2_calls']]

,job,marital,day,month,poutcome,loan_default_risk,is_long_cal,week,qtr,is_q2_calls
0,management,married,5,may,never contacted,1,0,week 1,Q2,1
1,technician,single,5,may,never contacted,1,0,week 1,Q2,1
2,entrepreneur,married,5,may,never contacted,1,0,week 1,Q2,1
3,blue-collar,married,5,may,never contacted,1,0,week 1,Q2,1
4,blue-collar,single,5,may,never contacted,0,0,week 1,Q2,1
...,...,...,...,...,...,...,...,...,...,...
44439,technician,married,17,nov,never contacted,0,1,week 3,Q4,0
44440,retired,divorced,17,nov,never contacted,0,1,week 3,Q4,0
44441,retired,married,17,nov,success,0,1,week 3,Q4,0
44442,blue-collar,married,17,nov,never contacted,0,1,week 3,Q4,0


In [62]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=0)

power_pipe = Pipeline([
    ('log_transformer',PowerTransformer()),
    ('scalling',StandardScaler())
    ])

preprocessor = ColumnTransformer([
            ('log_pipe',power_pipe,['age','balance']),

            ('cardinal',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),
             ['job','marital','day','month','poutcome','loan_default_risk' ,'is_long_cal' ,'week' ,'qtr' ,'is_q2_calls']),

            ('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),
              ['campaign','age_bin','balance_bins','duration_min','pdays_cat']),
              
            ('numerical',StandardScaler(),['duration','pdays'])
                                      ])


def objective(trial,preprocessor):
    classifier = trial.suggest_categorical('classifier',['CatBoostClassifier'])

 
    if classifier == 'CatBoostClassifier':
        params = {
            'iterations': trial.suggest_int('model__iterations',10,2000),
            'learning_rate': trial.suggest_float('model__learning_rate',0.001,0.3),
            'depth': trial.suggest_int('model__depth',1,16),
            'loss_function': trial.suggest_categorical('model__loss_function',['Logloss']),
            'l2_leaf_reg': trial.suggest_float('model__l2_leaf_reg',0.1,10),
            'random_strength': trial.suggest_float('model__random_strength',0,100),
            'bootstrap_type': trial.suggest_categorical('model__bootstrap_type',['MVS','Bayesian']),
            'grow_policy': trial.suggest_categorical('model__grow_policy',['Lossguide','SymmetricTree']),
            'od_type': trial.suggest_categorical('model__od_type',['IncToDec','Iter']),
            'od_wait': trial.suggest_int('model__od_wait',10,100),
            'one_hot_max_size': trial.suggest_int('model__one_hot_max_size',0,100),
           
        }
        

        pipe = Pipeline([
        ('preprocessor',preprocessor),
        ('model',CatBoostClassifier(**params,verbose=False,random_seed=0,auto_class_weights='Balanced'))
        ])
        

        training_model = pipe.fit(x_train,y_train)
        training_score = training_model.score(x_train,y_train)

        cv = cross_val_score(estimator=pipe,X=x_train,y=y_train,cv=StratifiedKFold(n_splits=10,shuffle=True,random_state=0),
                             scoring='roc_auc',n_jobs=-1)
        
        trial.set_user_attr('train_score', training_score)
        trial.set_user_attr('cv_mean', cv.mean())
        trial.set_user_attr('model', pipe)

        return cv.mean()
    
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=0),pruner=optuna.pruners.HyperbandPruner())
study.optimize(lambda trial:objective(trial,preprocessor),n_trials=10)

print(f'optuna best train roc_auc score : {study.best_trial.user_attrs["train_score"]}')
print(f'optuna best cv roc_auc score : {study.best_value}')
print(f'optuna best params : {study.best_params}')



final_model = study.best_trial.user_attrs['model']

final_model.fit(x_train,y_train)

y_pred = final_model.predict_proba(x_test)[:,1]

print(f'Test roc_auc score : {roc_auc_score(y_test,y_pred)}')
        


[I 2025-09-30 15:42:48,405] A new study created in memory with name: no-name-9c21fbfd-5671-4cb0-b838-e6b21066aa04
[I 2025-09-30 15:46:36,798] Trial 0 finished with value: 0.9191924816160301 and parameters: {'classifier': 'CatBoostClassifier', 'model__iterations': 1102, 'model__learning_rate': 0.2148416205453534, 'model__depth': 10, 'model__loss_function': 'Logloss', 'model__l2_leaf_reg': 5.494343511669279, 'model__random_strength': 42.36547993389047, 'model__bootstrap_type': 'MVS', 'model__grow_policy': 'SymmetricTree', 'model__od_type': 'Iter', 'model__od_wait': 58, 'model__one_hot_max_size': 57}. Best is trial 0 with value: 0.9191924816160301.
[I 2025-09-30 15:47:47,838] Trial 1 finished with value: 0.9207244208232241 and parameters: {'classifier': 'CatBoostClassifier', 'model__iterations': 1852, 'model__learning_rate': 0.022239781401168196, 'model__depth': 2, 'model__loss_function': 'Logloss', 'model__l2_leaf_reg': 0.30016213465922464, 'model__random_strength': 83.2619845547938, 'mo

optuna best train roc_auc score : 0.8869638588102939
optuna best cv roc_auc score : 0.9295984896586361
optuna best params : {'classifier': 'CatBoostClassifier', 'model__iterations': 1977, 'model__learning_rate': 0.031511398413660394, 'model__depth': 4, 'model__loss_function': 'Logloss', 'model__l2_leaf_reg': 1.696964227061463, 'model__random_strength': 65.31083254653984, 'model__bootstrap_type': 'Bayesian', 'model__grow_policy': 'Lossguide', 'model__od_type': 'Iter', 'model__od_wait': 22, 'model__one_hot_max_size': 19}
Test roc_auc score : 0.9293870269673804
